In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import numpy as np

from clonalg.antibody.real_antibody import RealAntibody, RealAntibodyBuilder
from clonalg.model.clonalg_optimization import OptimizationClonalg
from clonalg.problems.problem import Rastrigin
from clonalg.visualization.visualization import *

In [4]:
rastrigin = Rastrigin()
n_dims = 2
bounds = [(-4.5, 4.5)] * n_dims


def factory() -> RealAntibody:
    genes = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])
    return (
        RealAntibodyBuilder()
        .with_genes(genes)
        .with_bounds(bounds)
        .with_cost_fn(rastrigin)  # affinity = 1 / (1 + rastrigin(genes))
        .build()
    )


clonalg = OptimizationClonalg(
    population_size=50,
    clone_factor=0.2,
    n_generations=300,
    antibody_factory=factory,
    hypermutation_strategy="rank",
    rho=1.0,
    suppression_threshold=0.1,
)

memory = clonalg.run()
memory.sort(key=lambda ab: ab.affinity(None), reverse=True)

100%|██████████| 300/300 [00:10<00:00, 29.51it/s]


In [5]:
best = memory[0]
paths = list(map(lambda x: np.array(x.genes).reshape(1, 2), memory))

print(f"Best solution: x = {best.genes}")
print(f"f(x)         = {rastrigin(best.genes):.6f}")
print(f"affinity     = {best.affinity(None):.6f}")

Best solution: x = [-0.00335111  0.01755855]
f(x)         = 0.063331
affinity     = 0.940441


In [6]:
plot_3d_surface_without_grid(rastrigin)
plot_contour_and_paths(rastrigin, paths)

In [7]:
assert False

AssertionError: 

In [ ]:
import itertools
from dataclasses import dataclass, field


@dataclass
class TuningResult:
    params: dict
    score: float
    population: list = field(repr=False)


def tune_hyperparameters(
    param_grid: dict,
    n_runs: int = 3,
) -> tuple[TuningResult, list[TuningResult]]:
    """
    Grid search over CLONALG hyperparameters for the Rastrigin problem.

    Score = mean sum of rastrigin(ab.genes) over the final population across n_runs.
    Lower score means the population settled into better (lower) local minima.

    Args:
        param_grid: dict mapping hyperparameter name -> list of values to try.
        n_runs:     how many independent runs to average per combination (reduces noise).

    Returns:
        best: TuningResult with the lowest score.
        all_results: all TuningResults sorted from best to worst.
    """
    keys = list(param_grid.keys())
    combos = list(itertools.product(*param_grid.values()))
    print(
        f"Testing {len(combos)} combinations × {n_runs} runs = {len(combos) * n_runs} total runs\n"
    )

    all_results: list[TuningResult] = []

    for i, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))

        total_score = 0.0
        last_population = None
        for _ in range(n_runs):
            clonalg = OptimizationClonalg(antibody_factory=factory, **params)
            population = clonalg.run(verbose=False)
            total_score += sum(rastrigin(ab.genes) for ab in population)
            last_population = population

        avg_score = total_score / n_runs
        all_results.append(
            TuningResult(params=params, score=avg_score, population=last_population)
        )

        if i % 10 == 0 or i == len(combos):
            print(
                f"  [{i}/{len(combos)}] best so far: {min(all_results, key=lambda r: r.score).score:.4f}"
            )

    all_results.sort(key=lambda r: r.score)
    return all_results[0], all_results

In [ ]:
param_grid = {
    "population_size": [20],
    "clone_factor": [0.2],
    "n_replace": [5],
    "n_generations": [200],
    "rho": [1.0, 2.0, 3.0, 5.0],
    "suppression_threshold": [0.05, 0.1, 0.2],
}

best, all_results = tune_hyperparameters(param_grid, n_runs=3)

print(f"\nBest score : {best.score:.4f}")
print(f"Best params: {best.params}")

Testing 12 combinations × 3 runs = 36 total runs

  [10/12] best so far: 73.6548
  [12/12] best so far: 73.6548

Best score : 73.6548
Best params: {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho': 1.0, 'suppression_threshold': 0.05}


In [ ]:
print("Top 5 results:")
for r in all_results[:5]:
    print(f"  score={r.score:.4f}  {r.params}")

print("\nBottom 5 results:")
for r in all_results[-5:]:
    print(f"  score={r.score:.4f}  {r.params}")

Top 5 results:
  score=73.6548  {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho': 1.0, 'suppression_threshold': 0.05}
  score=86.9010  {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho': 2.0, 'suppression_threshold': 0.05}
  score=112.0542  {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho': 3.0, 'suppression_threshold': 0.05}
  score=125.8842  {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho': 5.0, 'suppression_threshold': 0.05}
  score=139.5237  {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho': 1.0, 'suppression_threshold': 0.1}

Bottom 5 results:
  score=169.7214  {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho': 5.0, 'suppression_threshold': 0.1}
  score=183.5894  {'population_size': 20, 'clone_factor': 0.2, 'n_replace': 5, 'n_generations': 200, 'rho'